In [ ]:
#YEARLY COGNITIVON DDIVERGENCE ANALYSIS - PER TEST ANALYSIS

"""
Per-test version of the yearly divergence (You et al style) analysis.

Same matched, per-window logistic design as the pooled version, but run
separately for each cognitive test so you get a divergence horizon per
cognitive domain. For each test x window we fit one matched logistic model:
is_case ~ z_score + age_at_test.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

output_dir = 'divergence_yearly_pertest'
os.makedirs(output_dir, exist_ok=True)

N_CONTROLS  = 10
AGE_CALIPER = 2.0
RNG_SEED    = 42

data = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')
data = data[data['test_name'] != 'PMem'].copy()
data = data.rename(columns={'Sex': 'sex', 'Edu': 'edu'})

n_prev = (data['prevalent'] == 1).sum()
data = data[data['prevalent'] == 0].copy()
print(f"Dropped {n_prev:,} prevalent observations")

test_direction = {
    'PMat': -1, 'RT': -1, 'FI': 1, 'NM': 1, 'SD': 1, 'TMA': -1,
    'TMB': -1, 'MP': 1, 'PAL': 1, 'PV': 1, 'TR': 1, 'BL': 1
}

data['z_score_adj'] = -(data['z_score'] * data['test_name'].map(test_direction))
data = data[data['time_to_event'] > 0].dropna(
    subset=['z_score_adj', 'age_at_test', 'time_to_event', 'event', 'sex']).copy()

tests = [t for t in data['test_name'].unique()]
windows = list(range(1, 26))

print(f"\nN = {data['eid'].nunique():,}  participants")
print(f"Tests: {tests}")

In [ ]:
#Build matched per-test, per-window dataset
"""
For each test, build the same matched case/control table as the pooled
analysis, windowed by years before diagnosis. Each test is treated
independently (a person can be a case in several tests' tables, using whichever
visit of that test falls in the window).
"""
rng = np.random.default_rng(RNG_SEED)
mid = 0

def build_matched(test_df):
    td = test_df.copy()
    td['window'] = np.floor(td['time_to_event']).astype(int) + 1
    td = td[td['window'].between(1, max(windows))]
    cases    = td[td['event'] == 1]
    controls = td[td['event'] == 0]
    rows = []
    for w in windows:
        cw = cases[cases['window'] == w]
        pool = (controls[controls['time_to_event'] >= (w - 1)]
                .sample(frac=1, random_state=RNG_SEED)
                .drop_duplicates('eid'))
        if len(cw) == 0 or len(pool) == 0:
            continue
        p_age, p_sex, p_idx = (pool['age_at_test'].values,
                               pool['sex'].values, pool.index.values)
        for _, case in cw.iterrows():
            elig = p_idx[(np.abs(p_age - case['age_at_test']) <= AGE_CALIPER)
                         & (p_sex == case['sex'])]
            if len(elig) == 0:
                continue
            take = rng.choice(elig, size=min(N_CONTROLS, len(elig)), replace=False)
            global mid
            mid += 1
            rows.append({'match_id': mid, 'window': w, 'is_case': 1,
                         'z_score': case['z_score_adj'],
                         'age_at_test': case['age_at_test'], 'eid': case['eid']})
            for ci in take:
                cr = controls.loc[ci]
                rows.append({'match_id': mid, 'window': w, 'is_case': 0,
                             'z_score': cr['z_score_adj'],
                             'age_at_test': cr['age_at_test'], 'eid': cr['eid']})
    out = pd.DataFrame(rows)
    out['test'] = test_df['test_name'].iloc[0]
    return out

matched_all = []
for test in tests:
    m = build_matched(data[data['test_name'] == test])
    if len(m):
        matched_all.append(m)
        print(f"{test:5s}: {len(m):>8,} matched rows, "
              f"{m['window'].nunique()} windows")
matched = pd.concat(matched_all, ignore_index=True)
print(f"\nTotal matched rows: {len(matched):,}")

In [ ]:
#Per-test, per-window logistic regression in R
"""
-i matched -i tests -i windows -o all_results
For each test x window: is_case ~ z_score + age_at_test (logistic).
"""
%load_ext rpy2.ipython

In [ ]:
%%R -i matched -i tests -i windows -o all_results

library(broom)
library(dplyr)
library(survival)

all_results <- list()
row_idx <- 1

for (test in tests) {
  td <- matched %>% filter(test == !!test)
  if (nrow(td) == 0) next
  cat("\nTest:", test, "\n")
  for (w in windows) {
    wd <- td %>% filter(window == w)
    n_cases <- sum(wd$is_case)
    if (n_cases < 5 || nrow(wd) < 2) next
    
    fit <- tryCatch(
      clogit(is_case ~ z_score + age_at_test + strata(match_id), data = wd),
      error = function(e) return(NULL))
    if (is.null(fit)) next
    
    tidy_fit <- tidy(fit, conf.int = TRUE, exponentiate = TRUE)
    z_row <- tidy_fit[tidy_fit$term == "z_score", ]
    all_results[[row_idx]] <- data.frame(
      test = test, window = w, n_total = nrow(wd), n_cases = n_cases,
      z_or = z_row$estimate, z_ci_low = z_row$conf.low,
      z_ci_high = z_row$conf.high, z_pval = z_row$p.value)
    row_idx <- row_idx + 1
  }
  cat("  Done\n")
}

all_results <- do.call(rbind, all_results)
cat("\nTotal models fit:", nrow(all_results), "\n")

In [ ]:
#Per-test divergence horizons and small-multiples figure
all_results = all_results.sort_values(['test', 'window']).reset_index(drop=True)

rows = []
for test, g in all_results.groupby('test'):
    sig_set = set(g.loc[g['z_ci_low'] > 1, 'window'])
    h = 0
    for w in sorted(g['window']):
        if w in sig_set:
            h = w
        else:
            break
    rows.append({'test': test, 'divergence_horizon_yrs': h,
                 'furthest_sig': max(sig_set) if sig_set else 0,
                 'max_window_fit': int(g['window'].max()),
                 'pattern': ''.join('*' if w in sig_set else '.'
                                    for w in sorted(g['window']))})
horizons = pd.DataFrame(rows).sort_values('divergence_horizon_yrs', ascending=False)
print(horizons.to_string(index=False))

all_results.to_csv(f'{output_dir}/divergence_results_pertest.csv', index=False)
horizons.to_csv(f'{output_dir}/divergence_horizons_pertest.csv', index=False)

tests_fit = sorted(all_results['test'].unique())
n = len(tests_fit); ncol = 4; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False)
for i, test in enumerate(tests_fit):
    ax = axes[i // ncol][i % ncol]
    g = all_results[all_results['test'] == test]
    ax.fill_between(g['window'], g['z_ci_low'], g['z_ci_high'], alpha=0.25, color='black')
    ax.plot(g['window'], g['z_or'], marker='o', markersize=3, color='black', linewidth=1.5)
    ax.axhline(y=1, color='darkgrey', linestyle='--', linewidth=1)
    ax.invert_xaxis()
    ax.set_title(test, fontsize=11)
    ax.set_xlabel('Years before dx', fontsize=9)
    ax.set_ylabel('OR / 1 SD worse', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
for j in range(n, nrow * ncol):
    axes[j // ncol][j % ncol].axis('off')
fig.suptitle('Per-test yearly divergence: when does each domain diverge?', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{output_dir}/divergence_pertest_grid.png', dpi=800, bbox_inches='tight')
plt.show()